# LeetCode 4000 — Largest Integer With Given Digit Sum bằng Python

Link đề: https://leetcode.com/problems/largest-integer-with-given-digit-sum/ · Độ khó: **Easy** · Chủ đề: Greedy, Math, Construction

Skeleton LeetCode:

```python
class Solution:
    def largestInteger(self, n: int, s: int) -> int:
        ...
```

> Ghi chú: tên bài, ví dụ, constraints và chữ ký hàm được đối chiếu qua leetcode.doocs.org (Easy, không Premium); phần phát biểu đề được viết lại bằng lời của mình. Slug URL được suy ra từ tên bài.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm **số nguyên không âm lớn nhất** có **tối đa `n` chữ số** và **tổng các chữ số bằng `s`**; nếu không có thì trả `-1`.
2. Nhận ra điều kiện tồn tại: cần `s <= 9 * n` (mỗi chữ số tối đa `9`).
3. Nắm **tham lam theo chữ số từ trái sang phải**: ở mỗi vị trí đặt chữ số lớn nhất có thể `min(9, còn_lại)`; kết quả có dạng `99..9 r 00..0`.
4. Viết 4 cách (vét cạn giảm dần, tham lam từng chữ số, công thức `divmod`, DP/đệ quy kiểm chứng) và đối chiếu bằng test vét cạn toàn miền `n <= 5`, `s <= 100`.
5. Tránh lỗi: đặt chữ số dư lên đầu (số bị nhỏ), quên trả `-1` khi `s > 9n`, quên ca `s = 0`, nhầm "đúng `n` chữ số" với "tối đa `n` chữ số".

## 2. Đề bài & ví dụ

Cho hai số nguyên không âm `n` và `s`. Hãy trả về **số nguyên lớn nhất** có **tối đa `n` chữ số** (không tính chữ số `0` đứng đầu) và **tổng các chữ số bằng `s`**. Nếu không tồn tại số như vậy, trả về `-1`.

| Ví dụ | `n`, `s` | Output | Giải thích |
|---|---|---|---|
| 1 | `n=2, s=9` | `90` | `90` là số lớn nhất có tối đa 2 chữ số và tổng chữ số bằng 9 (`9 + 0`) |
| 2 | `n=2, s=19` | `-1` | Hai chữ số cộng tối đa `18 < 19` nên không có số nào |
| 3 | `n=5, s=0` | `0` | Chỉ có số `0` có tổng chữ số bằng `0` |

Constraints: `1 <= n <= 5`; `0 <= s <= 100`.

Lưu ý "tối đa `n` chữ số": số `90` (2 chữ số) được so với các số 1 chữ số, như `9`; số lớn nhất sẽ là `90`.

In [1]:
def digit_sum(x):
    return sum(int(c) for c in str(x))

for n, s in [(2, 9), (2, 19), (5, 0)]:
    cands = [x for x in range(10 ** n) if digit_sum(x) == s]
    print(f"n={n}, s={s}: có {len(cands)} số, lớn nhất = {max(cands) if cands else -1}")

n=2, s=9: có 10 số, lớn nhất = 90
n=2, s=19: có 0 số, lớn nhất = -1
n=5, s=0: có 1 số, lớn nhất = 0


## 3. Kiến thức nền

### 3.1. Tổng các chữ số và `divmod`

`digit_sum(x)` có thể tính bằng `sum(int(c) for c in str(x))` hoặc vòng lặp `divmod(x, 10)`. Với `s` cho trước, ta cần biết `s` phân rã thế nào thành các chữ số `0..9`.

In [2]:
x, total = 8091, 0
while x:
    x, d = divmod(x, 10)
    total += d
print(total, sum(map(int, "8091")))

18 18


### 3.2. Phép chia nguyên và dư: `s // 9`, `s % 9`

`divmod(s, 9)` cho `(q, r)` với `s = 9 q + r`, `0 <= r < 9`. Ý nghĩa: `q` chữ số `9` cộng thêm một chữ số `r`. Ví dụ `divmod(20, 9) = (2, 2)`: `9 + 9 + 2`.

In [3]:
for s in (0, 9, 19, 20, 45):
    print(s, divmod(s, 9))

0 (0, 0)
9 (1, 0)
19 (2, 1)
20 (2, 2)
45 (5, 0)


### 3.3. Tạo số từ chuỗi chữ số, độ dài cố định

Chuỗi `"99" + "2" + "00"` là `"99200"`; `int("99200") = 99200`. `"9" * q` lặp chữ số `9`; `"0" * k` lặp chữ số `0`. Chữ số `0` đứng đầu bị bỏ khi đổi sang `int`: `int("0090") = 90`.

In [4]:
print("9" * 2 + str(2) + "0" * 2, int("99200"), int("0090"), int("000"))

99200 99200 90 0


### 3.4. So sánh số có cùng độ dài: so sánh từ trái sang phải

Hai số cùng độ dài chữ số (đệm `0` bên trái) so sánh theo từ điển từ chữ số đầu: chữ số đầu **lớn hơn** thì số đó lớn hơn. Vì vậy để cực đại hóa, ta làm chữ số trái nhất lớn nhất có thể, rồi tới chữ số kế, v.v. — tinh thần của **tham lam**.

In [5]:
print(max(["0900", "0990", "0909", "0099"]), 990 > 909 > 99)

0990 True


### 3.5. Giới hạn tồn tại: `s <= 9 * n`

`n` chữ số, mỗi chữ số `<= 9` nên tổng tối đa `9 n`. Nếu `s > 9 n` thì **không tồn tại** số nào. Ngược lại luôn tồn tại (đặt các `9` và phần dư). Đây là điều kiện kiểm tra đầu hàm.

In [6]:
for n in (1, 2, 5):
    print(n, 9 * n)

1 9
2 18
5 45


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Muốn số **lớn nhất**, ta muốn nó **dài nhất có thể** (đủ `n` chữ số) và các chữ số **bên trái lớn nhất có thể**. Mỗi chữ số tối đa `9`, nên đặt `9` ở đầu miễn là **tổng còn lại** đủ để dùng.

Tính tay `n = 2, s = 9`: chữ số đầu đặt `min(9, 9) = 9`, còn `0`; chữ số thứ hai đặt `min(9, 0) = 0` -> `90`.
Tính tay `n = 3, s = 20`: chữ số đầu `9` (còn 11), chữ số hai `9` (còn 2), chữ số ba `2` -> `992`.
Tính tay `n = 3, s = 5`: chữ số đầu `min(9, 5) = 5` (còn 0), hai chữ số sau `0` -> `500`.
Tính tay `n = 2, s = 19`: `9 * 2 = 18 < 19` -> `-1`.

| `n` | `s` | các chữ số (tham lam) | số |
|---|---|---|---|
| 2 | 9 | 9, 0 | `90` |
| 3 | 20 | 9, 9, 2 | `992` |
| 3 | 5 | 5, 0, 0 | `500` |
| 2 | 19 | (không tồn tại) | `-1` |

In [7]:
def greedy_digits(n, s):
    if s > 9 * n:
        return None
    digits, rem = [], s
    for _ in range(n):
        d = min(9, rem)
        digits.append(d)
        rem -= d
    return digits

for n, s in [(2, 9), (3, 20), (3, 5), (2, 19), (5, 0)]:
    ds = greedy_digits(n, s)
    print(n, s, ds, int("".join(map(str, ds))) if ds is not None else -1)

2 9 [9, 0] 90
3 20 [9, 9, 2] 992
3 5 [5, 0, 0] 500
2 19 None -1
5 0 [0, 0, 0, 0, 0] 0


## 5. Cách 1 — Brute Force / baseline: duyệt giảm dần từ `10^n - 1`

Số lớn nhất có tối đa `n` chữ số là `10^n - 1`. Duyệt `x` từ `10^n - 1` giảm dần về `0`; số đầu tiên có tổng chữ số bằng `s` chính là đáp án (vì ta xét từ lớn xuống nhỏ). Không có thì `-1`.

Với `n <= 5` chỉ tối đa `10^5` số nên chạy được, nhưng lãng phí: ta đang **tìm** thứ có thể **dựng** thẳng.

In [8]:
class SolutionBrute:
    def largestInteger(self, n, s):
        for x in range(10 ** n - 1, -1, -1):
            total, y = 0, x
            while y:
                y, d = divmod(y, 10)
                total += d
            if total == s:
                return x
        return -1

print(SolutionBrute().largestInteger(2, 9), SolutionBrute().largestInteger(2, 19), SolutionBrute().largestInteger(5, 0))

90 -1 0


**Độ phức tạp:** Time `O(10^n * n)`, Space `O(1)`. Với `n <= 5` còn chấp nhận được, nhưng sẽ vỡ nếu `n` lớn.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá (tham lam trái -> phải):** ở mỗi vị trí, đặt chữ số **lớn nhất có thể** là `min(9, tổng_còn_lại)`.

**Chứng minh tham lam:** giả sử có đáp án tối ưu mà chữ số đầu `a < min(9, s)`. Khi đó tồn tại chữ số sau nào đó lớn hơn `0` (vì tổng còn lại `s - a > 0` phải được chia cho các chữ số sau); ta chuyển bớt `1` từ chữ số sau đó cho chữ số đầu (chữ số đầu tăng `1`, chữ số sau giảm `1`): tổng không đổi, nhưng số **tăng** (chữ số đầu có trọng số lớn hơn). Mâu thuẫn tối ưu. Vậy chữ số đầu phải là `min(9, s)`, rồi áp dụng tương tự cho phần còn lại.

**Hệ quả (dạng đóng):** `q, r = divmod(s, 9)`: có `q` chữ số `9` đứng đầu, rồi chữ số `r`, rồi toàn `0`. Điều kiện tồn tại `s <= 9 n`.

**Invariant:** sau khi đặt `k` chữ số đầu, `tổng_còn_lại = s - (tổng k chữ số)`, và luôn `0 <= tổng_còn_lại <= 9 (n - k)` miễn ban đầu `s <= 9 n`.

## 7. Các cách tiếp theo

### Cách 2 — Tham lam từng chữ số

Lặp `n` lần, mỗi lần lấy `d = min(9, rem)`.

In [9]:
class SolutionGreedy:
    def largestInteger(self, n, s):
        if s > 9 * n:
            return -1
        res, rem = 0, s
        for _ in range(n):
            d = min(9, rem)
            res = res * 10 + d
            rem -= d
        return res

print(SolutionGreedy().largestInteger(2, 9), SolutionGreedy().largestInteger(2, 19), SolutionGreedy().largestInteger(5, 0))

90 -1 0


**Độ phức tạp:** Time `O(n)`, Space `O(1)`.

### Cách 3 — Công thức đóng với `divmod` và chuỗi

`q, r = divmod(s, 9)`; số gồm `q` chữ số `9`, một chữ số `r`, rồi `n - q - 1` chữ số `0`. Ca `q == n` (khi `s = 9n`, `r = 0`): chỉ `q` chữ số `9`, không cần chữ số `r`. Để gọn: `("9" * q + str(r)).ljust(n, "0")` rồi cắt đúng `n` chữ số.

In [10]:
class SolutionFormula:
    def largestInteger(self, n, s):
        if s > 9 * n:
            return -1
        q, r = divmod(s, 9)
        digits = ("9" * q + str(r)).ljust(n, "0")[:n]       # q chữ số 9, chữ số dư r, rồi các số 0
        return int(digits)

print(SolutionFormula().largestInteger(2, 9), SolutionFormula().largestInteger(2, 19), SolutionFormula().largestInteger(5, 0))

90 -1 0


**Độ phức tạp:** Time `O(n)`, Space `O(n)` (chuỗi). Chú ý `[:n]` quan trọng khi `q = n` (`s = 9n`): chuỗi `"9" * n + "0"` dài `n + 1` nên cắt lại còn `n` chữ số.

### Cách 4 — Đệ quy/DP kiểm chứng (độc lập với tham lam)

`best(k, rem)` = số lớn nhất xây từ `k` chữ số còn lại với tổng `rem` (hoặc `None` nếu không được). Thử chữ số đầu `d` từ `9` xuống `0`, lấy phương án đầu tiên khả thi. Cách này **không** dựa trên chứng minh tham lam nên làm trọng tài thứ hai (độ phức tạp `O(n * s * 10)` nhờ ghi nhớ).

In [11]:
from functools import lru_cache

class SolutionDP:
    def largestInteger(self, n, s):
        @lru_cache(maxsize=None)
        def best(k, rem):
            # trả về chuỗi k chữ số lớn nhất với tổng rem, hoặc None
            if k == 0:
                return "" if rem == 0 else None
            for d in range(9, -1, -1):
                if d > rem:
                    continue
                tail = best(k - 1, rem - d)
                if tail is not None:
                    return str(d) + tail
            return None

        r = best(n, s)
        return -1 if r is None else int(r)

print(SolutionDP().largestInteger(2, 9), SolutionDP().largestInteger(2, 19), SolutionDP().largestInteger(5, 0))

90 -1 0


**Độ phức tạp:** Time `O(n * s * 10)` (số trạng thái `(k, rem)` nhân 10 lựa chọn), Space `O(n * s)`.

## 8. TRACE từng bước

Trace Cách 2 với `n = 3`, `s = 20`:

| vị trí | `rem` trước | `d = min(9, rem)` | `rem` sau | `res` sau |
|---|---|---|---|---|
| 1 | 20 | 9 | 11 | 9 |
| 2 | 11 | 9 | 2 | 99 |
| 3 | 2 | 2 | 0 | 992 |

Với `n = 2, s = 19`: `19 > 9 * 2 = 18` -> `-1` ngay.

In [12]:
def trace_greedy(n, s):
    print(f"n={n}, s={s}, giới hạn 9n={9 * n}")
    if s > 9 * n:
        print("  s > 9n -> -1")
        return -1
    res, rem = 0, s
    for pos in range(1, n + 1):
        d = min(9, rem)
        res = res * 10 + d
        print(f"  vị trí {pos}: rem={rem}, d=min(9,{rem})={d}, rem sau={rem - d}, res={res}")
        rem -= d
    return res

assert trace_greedy(3, 20) == 992
assert trace_greedy(2, 19) == -1
assert trace_greedy(2, 9) == 90
assert trace_greedy(5, 0) == 0

n=3, s=20, giới hạn 9n=27
  vị trí 1: rem=20, d=min(9,20)=9, rem sau=11, res=9
  vị trí 2: rem=11, d=min(9,11)=9, rem sau=2, res=99
  vị trí 3: rem=2, d=min(9,2)=2, rem sau=0, res=992
n=2, s=19, giới hạn 9n=18
  s > 9n -> -1
n=2, s=9, giới hạn 9n=18
  vị trí 1: rem=9, d=min(9,9)=9, rem sau=0, res=9
  vị trí 2: rem=0, d=min(9,0)=0, rem sau=0, res=90
n=5, s=0, giới hạn 9n=45
  vị trí 1: rem=0, d=min(9,0)=0, rem sau=0, res=0
  vị trí 2: rem=0, d=min(9,0)=0, rem sau=0, res=0
  vị trí 3: rem=0, d=min(9,0)=0, rem sau=0, res=0
  vị trí 4: rem=0, d=min(9,0)=0, rem sau=0, res=0
  vị trí 5: rem=0, d=min(9,0)=0, rem sau=0, res=0


In [13]:
def trace_formula(n, s):
    q, r = divmod(s, 9)
    print(f"s={s}: divmod(s, 9) = ({q}, {r}) -> {q} chữ số 9, chữ số dư {r}, còn lại là 0")
    return int(("9" * q + str(r)).ljust(n, "0")[:n]) if s <= 9 * n else -1

assert trace_formula(3, 20) == 992 and trace_formula(3, 27) == 999 and trace_formula(3, 0) == 0

s=20: divmod(s, 9) = (2, 2) -> 2 chữ số 9, chữ số dư 2, còn lại là 0
s=27: divmod(s, 9) = (3, 0) -> 3 chữ số 9, chữ số dư 0, còn lại là 0
s=0: divmod(s, 9) = (0, 0) -> 0 chữ số 9, chữ số dư 0, còn lại là 0


## 9. Code nộp LeetCode

Dùng Cách 2 (tham lam từng chữ số, dễ chứng minh).

In [14]:
from typing import List, Optional


class Solution:
    def largestInteger(self, n: int, s: int) -> int:
        if s > 9 * n:
            return -1
        res, rem = 0, s
        for _ in range(n):
            d = min(9, rem)
            res = res * 10 + d
            rem -= d
        return res


print(Solution().largestInteger(2, 9), Solution().largestInteger(2, 19), Solution().largestInteger(5, 0))

90 -1 0


## 10. Vì sao thuật toán đúng

1. **Tồn tại.** Số có tối đa `n` chữ số là chuỗi `n` chữ số (cho phép `0` đứng đầu), mỗi chữ số trong `[0, 9]`, nên tổng nằm trong `[0, 9n]`. Mọi giá trị trong khoảng đó đạt được (đặt các `9` và phần dư). Vậy tồn tại iff `s <= 9n` (với `s >= 0`).
2. **Lớn nhất = lớn nhất theo từ điển trên chuỗi `n` chữ số.** Cùng độ dài `n` (đệm `0` bên trái), số lớn hơn iff chuỗi lớn hơn theo từ điển.
3. **Chữ số đầu tối ưu.** Đặt `d1 = min(9, s)`. Giả sử đáp án tối ưu có `d1' < d1`. Khi đó `s - d1' > s - d1 >= 0` còn phải phân bổ cho `n - 1` chữ số sau; vì `d1' < min(9, s)` nên `d1' < 9` và `s - d1' >= 1`, nên có một chữ số sau `> 0`. Chuyển `1` từ chữ số sau đó sang chữ số đầu: hợp lệ (chữ số đầu `<= 9`, chữ số sau `>= 0`), tổng không đổi, số tăng. Mâu thuẫn.
4. **Khả thi phần còn lại.** Sau khi đặt `d1 = min(9, s)`, tổng còn lại `s - d1` thỏa `0 <= s - d1 <= 9(n-1)` (nếu `s <= 9`, còn `0`; nếu `s > 9`, `s - 9 <= 9n - 9`), nên bài toán con lại hợp lệ; áp dụng quy nạp.
5. **Cách 3 (công thức)** chính là kết quả của quá trình tham lam; **Cách 1, Cách 4** tìm bằng cách khác (vét cạn, DP) nên dùng để đối chiếu.

## 11. Phân tích Time / Space Complexity

- Cách 1 (vét cạn): Time `O(10^n * n)`, Space `O(1)`.
- Cách 2 (nộp): Time `O(n)`, Space `O(1)`.
- Cách 3 (công thức): Time `O(n)`, Space `O(n)` cho chuỗi.
- Cách 4 (DP): Time `O(n * s * 10)`, Space `O(n * s)`.

`n <= 5` nên mọi cách đều rất nhanh; điều quan trọng là **chứng minh tham lam** để biết vì sao nó đúng với mọi `n`.

## 12. Bộ test

### Cách kiểm thử trong notebook này

Chúng ta dùng ba lớp kiểm thử, từ dễ đến khó:

1. **Test cố định**: ví dụ của đề + các trường hợp biên tự nghĩ ra. Mỗi test là một `assert hàm(input) == kết_quả_mong_đợi`. Nếu sai, Python ném `AssertionError` và dừng ngay, kèm thông tin ta in ra để dễ lần lại.
2. **Test ngẫu nhiên có `random.seed`**: gọi `random.seed(số)` trước vòng lặp để **mỗi lần chạy lại sinh đúng cùng dữ liệu** (tái lập được). Input luôn được sinh **trong đúng constraints** của đề.
3. **Đối chiếu nhiều cách (oracle)**: một cách rất đơn giản, hiển nhiên đúng (brute force) đóng vai "trọng tài"; mọi cách nhanh hơn phải cho cùng kết quả với trọng tài trên mọi input ngẫu nhiên.

In [15]:
import random

solutions = [SolutionGreedy(), SolutionFormula(), SolutionDP(), Solution()]

fixed = [
    (2, 9, 90), (2, 19, -1), (5, 0, 0), (1, 9, 9), (1, 10, -1), (1, 0, 0), (3, 20, 992),
    (3, 5, 500), (3, 27, 999), (3, 28, -1), (5, 45, 99999), (5, 46, -1), (5, 100, -1),
    (4, 18, 9900), (4, 19, 9910), (2, 18, 99), (5, 1, 10000),
]
for n, s, expected in fixed:
    for sol in solutions + [SolutionBrute()]:
        assert sol.largestInteger(n, s) == expected, (type(sol).__name__, n, s)
print("test cố định OK:", len(fixed), "ca")

# vét cạn toàn miền của đề: n = 1..5, s = 0..100 (trọng tài: SolutionBrute)
for n in range(1, 6):
    for s in range(0, 101):
        expected = SolutionBrute().largestInteger(n, s)
        for sol in solutions:
            assert sol.largestInteger(n, s) == expected, (type(sol).__name__, n, s)
print("vét cạn toàn miền n=1..5, s=0..100 OK")

random.seed(4000)
for _ in range(500):
    n, s = random.randint(1, 5), random.randint(0, 100)
    assert len({sol.largestInteger(n, s) for sol in solutions}) == 1
print("test ngẫu nhiên OK")

test cố định OK: 17 ca


vét cạn toàn miền n=1..5, s=0..100 OK
test ngẫu nhiên OK


### Test thuộc tính (property test)

(1) Nếu kết quả `!= -1` thì tổng chữ số của nó bằng `s` và nó `< 10^n`. (2) `-1` khi và chỉ khi `s > 9n`. (3) Đơn điệu theo `n`: với `s` cố định, `largestInteger(n + 1, s) >= largestInteger(n, s)` khi cả hai hợp lệ. (4) Không có số nào lớn hơn kết quả mà vẫn thỏa điều kiện (kiểm bằng vét cạn).

In [16]:
for n in range(1, 6):
    for s in range(0, 101):
        r = Solution().largestInteger(n, s)
        assert (r == -1) == (s > 9 * n)
        if r != -1:
            assert digit_sum(r) == s and 0 <= r < 10 ** n
            assert all(digit_sum(x) != s for x in range(r + 1, 10 ** n))
        if n < 5 and r != -1:
            assert Solution().largestInteger(n + 1, s) >= r
print("property test OK")

property test OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: đặt chữ số **dư** lên đầu thay vì chữ số `9`

Với `s = 10`, `n = 3`: viết `r = 1` trước rồi `9` cho `190`, nhưng đáp án đúng `910` lớn hơn.

In [17]:
def wrong_remainder_first(n, s):
    if s > 9 * n:
        return -1
    q, r = divmod(s, 9)
    return int((str(r) + "9" * q).ljust(n, "0")[:n])          # SAI: chữ số nhỏ đứng trước các số 9

print(wrong_remainder_first(3, 10), "!=", Solution().largestInteger(3, 10))
assert wrong_remainder_first(3, 10) == 190 and Solution().largestInteger(3, 10) == 910

190 != 910


### Lỗi 2: quên trả `-1` khi `s > 9n`

Không kiểm tra tồn tại, tham lam vẫn chạy và trả số có tổng chữ số **nhỏ hơn** `s` (vì hết chỗ đặt).

In [18]:
def wrong_no_check(n, s):
    res, rem = 0, s
    for _ in range(n):
        d = min(9, rem)
        res = res * 10 + d
        rem -= d
    return res                                                # SAI: không kiểm tra rem còn dư

print(wrong_no_check(2, 19), "!=", Solution().largestInteger(2, 19))
assert wrong_no_check(2, 19) == 99 and Solution().largestInteger(2, 19) == -1 and digit_sum(99) != 19

99 != -1


### Lỗi 3: dùng toàn `9` (`10^n - 1`) hoặc `s` chữ số `9` bất kể tổng

Số gồm toàn chữ số `9` chỉ đúng khi `s = 9n`.

In [19]:
def wrong_all_nines(n, s):
    return 10 ** n - 1 if s <= 9 * n else -1                  # SAI: bỏ qua tổng

print(wrong_all_nines(2, 9), "!=", Solution().largestInteger(2, 9))
assert wrong_all_nines(2, 9) == 99 and Solution().largestInteger(2, 9) == 90

99 != 90


### Lỗi 4: nhầm "tối đa `n` chữ số" thành "đúng `n` chữ số" không cho `s = 0`

Với `s = 0` đáp án là `0` (có `1` chữ số). Nếu ép số có chữ số đầu khác `0`, `s = 0` bị coi là không tồn tại và trả `-1` sai.

In [20]:
def wrong_exact_digits(n, s):
    r = Solution().largestInteger(n, s)
    if r != -1 and len(str(r)) != n:                         # SAI: ép đúng n chữ số
        return -1
    return r

print(wrong_exact_digits(5, 0), "!=", Solution().largestInteger(5, 0))
assert wrong_exact_digits(5, 0) == -1 and Solution().largestInteger(5, 0) == 0
assert wrong_exact_digits(2, 9) == 90

-1 != 0


### Lỗi 5: chữ số dư sai chỗ khi `s = 9n` (thừa một chữ số)

Công thức `("9" * q + str(r)).ljust(n, "0")` không cắt, với `s = 9n` ta có `q = n`, `r = 0` nên chuỗi dài `n + 1` (có thêm `0`), cho số **lớn gấp 10 lần**.

In [21]:
def wrong_no_trim(n, s):
    if s > 9 * n:
        return -1
    q, r = divmod(s, 9)
    return int(("9" * q + str(r)).ljust(n, "0"))              # SAI: không cắt [:n]

print(wrong_no_trim(3, 27), "!=", Solution().largestInteger(3, 27))
assert wrong_no_trim(3, 27) == 9990 and Solution().largestInteger(3, 27) == 999

9990 != 999


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Số **lớn nhất** có tổng chữ số cho trước / độ dài tối đa" -> **tham lam từ chữ số cao nhất**, mỗi chữ số `min(9, còn_lại)`. "Nhỏ nhất" thì ngược lại: dồn tổng vào các chữ số **thấp** (đặt `9` ở cuối).
- Dạng đóng: `9...9 r 0...0` với `(q, r) = divmod(s, 9)`.
- Điều kiện tồn tại: `s <= 9 * n` (và `s >= 0`; nếu yêu cầu chữ số đầu khác `0` thì còn điều kiện `s >= 1`).
- Với `n` nhỏ, **vét cạn toàn miền** để kiểm công thức: thủ thuật kiểm chứng tốt nhất cho bài xây dựng.
- Chứng minh tham lam bằng **trao đổi (exchange argument)**: chuyển 1 đơn vị từ chữ số thấp lên cao thì số tăng.
- Đọc kỹ "tối đa `n` chữ số" so với "đúng `n` chữ số".

## 15. Biến thể của bài

1. **Smallest Integer With Given Digit Sum** (biến thể nhỏ nhất): dồn tổng ra cuối.
2. **Largest Number** (179), **Maximum Swap** (670): tham lam trên chữ số.
3. **Monotone Increasing Digits** (738), **Remove K Digits** (402): tham lam với chữ số.
4. **Construct the Lexicographically Largest Valid Sequence** (1718).
5. Biến thể: yêu cầu chữ số **đôi một khác nhau** (khó hơn: tối đa 9 chữ số `9,8,7,...`).
6. Biến thể: yêu cầu số **chia hết cho 3 / 9** hay tổng chữ số thuộc tập cho trước.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn giảm dần | thử từ `10^n - 1` xuống `0` | `O(10^n n)` | `O(1)` | `n` nhỏ, trọng tài |
| 2. Tham lam từng chữ số | `min(9, rem)` | `O(n)` | `O(1)` | **Nộp bài** |
| 3. Công thức `divmod` | `9..9 r 0..0` | `O(n)` | `O(n)` | Viết nhanh, dạng đóng |
| 4. DP/đệ quy | thử chữ số đầu `9..0` | `O(n s 10)` | `O(n s)` | Kiểm chứng độc lập, ràng buộc phức tạp |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().largestInteger(4, 23)` và `Solution().largestInteger(3, 3)` ra gì?

**Bài 2 (tính tay):** với `n = 4`, `s = 30`: viết từng chữ số theo tham lam và kết quả.

**Bài 3 (điền TODO):** hoàn thành `smallest_integer(n, s)`: số **nhỏ nhất** có **đúng** `n` chữ số (chữ số đầu khác 0) và tổng chữ số `s`, hoặc `-1`.

**Bài 4 (tự viết lại):** viết `count_integers(n, s)` đếm có bao nhiêu số (tối đa `n` chữ số) có tổng chữ số `s` (DP hoặc vét cạn).

**Bài 5 (mini challenge):** viết `largest_distinct_digits(s)`: số lớn nhất có các chữ số **đôi một khác nhau** và tổng `s` (hoặc `-1`).

In [22]:
# Bài 1: đoán rồi chạy
print(Solution().largestInteger(4, 23), Solution().largestInteger(3, 3))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def smallest_integer_todo(n, s):
    # TODO: chữ số đầu >= 1; dồn phần lớn tổng vào các chữ số cuối
    return -1

9950 300


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: `23 = 9 + 9 + 5`, bốn chữ số. `s = 3`, `n = 3`: chữ số đầu là `3`. Bài 2: `30 = 9 + 9 + 9 + 3`.

**Mức 2 (vừa):** Bài 3: chữ số đầu `1`, còn `s - 1` chia cho `n - 1` chữ số cuối, dồn `9` về **cuối** (phải có `s - 1 <= 9 (n - 1)`). Bài 4: DP `ways[k][t]`. Bài 5: duyệt chữ số `9, 8, 7, ...` đến khi hết tổng.

**Mức 3 (gần đáp án):** Bài 1: `9950` và `300`. Bài 2: `9993`. Bài 3: `smallest_integer(3, 20) = 299`, `smallest_integer(2, 0) = -1`. Bài 5: `largest_distinct_digits(17) = 98`, `largest_distinct_digits(46) = -1` (vì `9+8+...+1 = 45`).

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1, 2
assert Solution().largestInteger(4, 23) == 9950 and Solution().largestInteger(3, 3) == 300
assert greedy_digits(4, 30) == [9, 9, 9, 3] and Solution().largestInteger(4, 30) == 9993

# Bài 3
def smallest_integer(n, s):
    if s < 1 or s > 9 * n:
        return -1
    digits, rem = [], s
    first = max(1, rem - 9 * (n - 1))               # chữ số đầu nhỏ nhất sao cho phần còn lại vẫn khả thi
    digits.append(first)
    rem -= first
    for k in range(n - 1, 0, -1):
        d = max(0, rem - 9 * (k - 1))
        digits.append(d)
        rem -= d
    return int("".join(map(str, digits)))

assert smallest_integer(3, 20) == 299 and smallest_integer(2, 0) == -1 and smallest_integer(1, 5) == 5
assert smallest_integer(3, 1) == 100 and smallest_integer(2, 19) == -1
for n in range(1, 4):
    for s in range(0, 30):
        cands = [x for x in range(10 ** (n - 1), 10 ** n) if digit_sum(x) == s]
        assert smallest_integer(n, s) == (min(cands) if cands else -1), (n, s)

# Bài 4
def count_integers(n, s):
    ways = [1] + [0] * s                              # ways[t]: số cách chọn các chữ số cho tổng t
    for _ in range(n):
        new = [0] * (s + 1)
        for t in range(s + 1):
            if ways[t]:
                for d in range(10):
                    if t + d <= s:
                        new[t + d] += ways[t]
        ways = new
    return ways[s]

assert count_integers(2, 9) == 10 and count_integers(2, 19) == 0 and count_integers(5, 0) == 1
assert all(count_integers(3, s) == sum(1 for x in range(1000) if digit_sum(x) == s) for s in range(0, 30))

# Bài 5
def largest_distinct_digits(s):
    digits, rem = [], s
    for d in range(9, -1, -1):
        if rem >= d and rem > 0:
            digits.append(d)
            rem -= d
    return int("".join(map(str, digits))) if rem == 0 and digits else (0 if s == 0 else -1)

assert largest_distinct_digits(17) == 98 and largest_distinct_digits(46) == -1 and largest_distinct_digits(0) == 0
assert largest_distinct_digits(45) == 987654321
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

### Template

```python
class Solution:
    def largestInteger(self, n: int, s: int) -> int:
        if s > 9 * n:                 # không đủ chỗ: mỗi chữ số tối đa 9
            return -1
        res, rem = 0, s
        for _ in range(n):
            d = min(9, rem)           # tham lam: chữ số lớn nhất có thể, từ trái sang phải
            res = res * 10 + d
            rem -= d
        return res
```

Khung chung "số lớn nhất có tổng chữ số cho trước": **dồn 9 về bên trái**; dạng đóng `9..9 r 0..0`.

### Checklist

- [ ] Đã đọc kỹ constraints (độ dài, miền giá trị) và biết thuật toán chạy được trong giới hạn đó.
- [ ] Đã chạy tay ít nhất **một ví dụ của đề** và một **trường hợp biên** (nhỏ nhất, lớn nhất, giá trị đặc biệt).
- [ ] Đúng **tên class / tên hàm / kiểu trả về** như skeleton LeetCode (đúng `bool`, `int`, `list`...).
- [ ] Không dùng `input()`, không `print` thừa trong code nộp.
- [ ] Không làm hỏng dữ liệu đầu vào nếu đề không cho phép (sao chép trước khi sửa).
- [ ] Đã so sánh với một cách brute force trên input ngẫu nhiên nhỏ.
- [ ] Đã xử lý `s > 9n` (trả `-1`) và `s = 0` (trả `0`), và dựng số từ trái sang phải với `min(9, rem)`.
- [ ] Đã kiểm bằng vét cạn toàn miền `n <= 5`, `s <= 100` (hoặc ít nhất vài trăm cặp).